In [7]:
# Find cases where annotators disagree on labels, and save them to a CSV.
# - Align items by canonicalized text intersection across the three files.
# - Normalize labels to {positive, negative, neutral}.
# - Output rows where at least one annotator differs.
# - Save to ./label_disagreements.csv

import unicodedata
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np

PATH_THANH  = Path("./Thanh.csv")
PATH_TRANG  = Path("./Trang.csv")
PATH_TRUONG = Path("./Truong.csv")

CATEGORIES = ["positive", "negative", "neutral"]

def canonicalize_text(t: str):
    if t is None:
        return None
    t = unicodedata.normalize("NFC", str(t))
    t = " ".join(t.split())
    return t

def normalize_label(x: str):
    if x is None:
        return None
    s = str(x).strip().lower()
    mapping = {
        # dương
        "pos": "positive", "positive": "positive", "+": "positive", "1": "positive",
        # âm
        "neg": "negative", "negative": "negative", "-": "negative", "-1": "negative", "2": "negative",
        # trung tính
        "neu": "neutral",  "neutral": "neutral",  "0": "neutral"
    }
    return mapping.get(s, None)

def to_df_from_csv(path: Path, annotator_name: str):
    """
    Đọc CSV và chuẩn hoá về 3 cột: text_raw, text_canon, annotator_name.
    Giả định tối thiểu có cột label; text có thể là text_canon hoặc text/raw khác.
    """
    df = pd.read_csv(path)

    # 1) Xác định cột text
    text_col = None
    for cand in ["text_canon", "text_raw", "text", "content", "body", "sentence", "review"]:
        if cand in df.columns:
            text_col = cand
            break

    if text_col is None:
        # không có cột text rõ ràng → gộp tất cả cột thành 1 chuỗi
        df["text_raw"] = df.astype(str).agg(" ".join, axis=1)
    else:
        df["text_raw"] = df[text_col].astype(str)

    # canonical
    df["text_canon"] = df["text_raw"].apply(canonicalize_text)

    # 2) Xác định cột label
    label_col = None
    for cand in ["label", "sentiment", "y", "tag", "prediction"]:
        if cand in df.columns:
            label_col = cand
            break

    if label_col is None:
        df[annotator_name] = None
    else:
        df[annotator_name] = df[label_col].apply(normalize_label)

    return df[["text_raw", "text_canon", annotator_name]]

# === Load + convert từ CSV ===
df_thanh  = to_df_from_csv(PATH_THANH,  "Thanh")
df_trang  = to_df_from_csv(PATH_TRANG,  "Trang")
df_truong = to_df_from_csv(PATH_TRUONG, "Truong")

# === Aggregate by text_canon cho từng annotator (nếu trùng → lấy mode, hoà → NaN) ===
def pick_mode_or_nan(series):
    vals = [v for v in series if pd.notna(v)]
    if not vals:
        return np.nan
    c = Counter(vals).most_common()
    if len(c) == 1:
        return c[0][0]
    if len(c) >= 2 and c[0][1] == c[1][1]:
        return np.nan  # tie
    return c[0][0]

agg_thanh  = df_thanh.groupby("text_canon", as_index=False).agg(
    text_raw=("text_raw", "first"),
    Thanh=("Thanh", pick_mode_or_nan)
)
agg_trang  = df_trang.groupby("text_canon", as_index=False).agg(
    text_raw=("text_raw", "first"),
    Trang=("Trang", pick_mode_or_nan)
)
agg_truong = df_truong.groupby("text_canon", as_index=False).agg(
    text_raw=("text_raw", "first"),
    Truong=("Truong", pick_mode_or_nan)
)

# === Lấy giao nhau của 3 annotator theo text_canon ===
common = (
    agg_thanh
    .merge(agg_trang[["text_canon", "Trang"]], on="text_canon", how="inner")
    .merge(agg_truong[["text_canon", "Truong"]], on="text_canon", how="inner")
)

# Bỏ những dòng thiếu nhãn (NaN do tie hoặc thiếu label)
common = common.dropna(subset=["Thanh", "Trang", "Truong"])

# === Giữ lại những dòng có bất đồng nhãn ===
mask_disagree = ~(
    (common["Thanh"] == common["Trang"]) &
    (common["Trang"] == common["Truong"])
)
disagreements = common.loc[mask_disagree].copy()

# === Thêm majority_vote (đa số) ===
def majority_vote(row):
    labels = [row["Thanh"], row["Trang"], row["Truong"]]
    cnt = Counter(labels).most_common()
    if len(cnt) == 0:
        return None
    if len(cnt) > 1 and cnt[0][1] == cnt[1][1]:
        return "tie"
    return cnt[0][0]

disagreements["majority_vote"] = disagreements.apply(majority_vote, axis=1)

# === Lưu ra CSV ===
out_path = "./conflicts_all.csv"
disagreements.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"Số văn bản chung (đủ 3 nhãn): {len(common)}")
print(f"Số trường hợp bất đồng nhãn: {len(disagreements)}")
print("Đã lưu CSV:", out_path)

# Preview 10 disagreements
print("\n=== Preview 10 disagreements ===")
cols_to_show = ["text_raw", "Thanh", "Trang", "Truong", "majority_vote"]
for i, row in disagreements.head(10).iterrows():
    print("---")
    txt = row["text_raw"]
    print("text_raw:", (txt[:300] + ("..." if len(txt) > 300 else "")))
    print("Thanh:", row["Thanh"],
          "| Trang:", row["Trang"],
          "| Truong:", row["Truong"],
          "| majority:", row["majority_vote"])

Số văn bản chung (đủ 3 nhãn): 155
Số trường hợp bất đồng nhãn: 84
Đã lưu CSV: ./conflicts_all.csv

=== Preview 10 disagreements ===
---
text_raw: Bác sĩ Kỳ Y Dược chia sẻ lưu ý để nâng mũi đẹp, an toàn — Theo ThS.BS Nguyễn Trường Kỳ (thường gọi là bác sĩ Kỳ Y Dược) - khách hàng nên chọn chuyên gia, tìm hiểu về các ca thẩm mỹ hiệu quả để quyết định phương pháp nâng mũi phù hợp với bản thân.
Thanh: positive | Trang: neutral | Truong: positive | majority: positive
---
text_raw: Bé trai trong vụ BVĐK tỉnh Nam Định bị tố 'nộp đủ tiền mới cấp cứu' xuất viện — Bé trai 4 tuổi trong vụ Bệnh viện Đa khoa tỉnh Nam Định bị tố "nộp đủ tiền mới cấp cứu" ra viện sáng nay, mọi chi phí điều trị được BHYT chi trả.
Thanh: neutral | Trang: positive | Truong: negative | majority: tie
---
text_raw: Bình Định: 5 năm, thêm 7.500 nhân sự ngành bán dẫn, AI, an ninh mạng — UBND tỉnh Bình Định vừa phê duyệt Đề án Phát triển nguồn nhân lực ngành công nghiệp bán dẫn, trí tuệ nhân tạo, an toàn và an ninh mạng tỉnh B

In [8]:
# === Các dòng mà 3 annotator đồng thuận tuyệt đối ===
mask_agree = (
    (common["Thanh"] == common["Trang"]) &
    (common["Trang"] == common["Truong"])
)

agreements = common.loc[mask_agree].copy()
agreements["agreed_label"] = agreements["Thanh"]  # vì cả 3 giống nhau

out_agree_path = "./non_conflict_all.csv"
agreements.to_csv(out_agree_path, index=False, encoding="utf-8-sig")

print(f"Số trường hợp đồng thuận tuyệt đối: {len(agreements)}")
print("Đã lưu file đồng thuận:", out_agree_path)


Số trường hợp đồng thuận tuyệt đối: 71
Đã lưu file đồng thuận: ./non_conflict_all.csv


Fill các dòng được gán đúng vào file label_disagreements_fixed.csv

In [2]:
import pandas as pd

# Đường dẫn file
file_all  = "./label_disagreements_fixed.csv"
file_sub  = "./conflicts_all.csv"
out_file  = "./label_disagreements_fixed.csv"

# Đọc dữ liệu
df_all = pd.read_csv(file_all)
df_sub = pd.read_csv(file_sub)

# 1) Chọn cột khóa
if "review" in df_all.columns and "review" in df_sub.columns:
    KEY_COL = "review"
elif "review" in df_all.columns and "text_raw" in df_sub.columns:
    KEY_COL = "review"
else:
    raise ValueError("Không tìm được cột khóa chung (text_canon / text_raw). Hãy chỉnh KEY_COL thủ công.")

# 2) Xác định các cột nhãn cần fill
#    - bỏ cột text, chỉ giữ các cột giống nhau giữa 2 file
exclude_cols = {KEY_COL, "text_raw"}
label_cols_sub = [c for c in df_sub.columns if c not in exclude_cols]
label_cols = [c for c in label_cols_sub if c in df_all.columns]

print("Cột khóa:", KEY_COL)
print("Các cột sẽ được fill từ non_conflict_all.csv:", label_cols)

# 3) Fill nhãn cho từng cột
for col in label_cols:
    # mapping: key -> label từ file con
    sub_map = df_sub.set_index(KEY_COL)[col]

    # chỉ fill vào những chỗ đang NaN và có key trong mapping
    mask = df_all[col].isna() & df_all[KEY_COL].isin(sub_map.index)

    before_count = mask.sum()
    df_all.loc[mask, col] = df_all.loc[mask, KEY_COL].map(sub_map)
    after_count = df_all[col].notna().sum()

    print(f"Cột {col}: số ô được fill thêm = {before_count}")

# 4) Ghi ra file mới
df_all.to_csv(out_file, index=False, encoding="utf-8-sig")
print(f"Đã ghi file: {out_file}")


Cột khóa: review
Các cột sẽ được fill từ non_conflict_all.csv: ['sentiment']
Cột sentiment: số ô được fill thêm = 84
Đã ghi file: ./label_disagreements_fixed.csv
